In [1]:
import os
import cv2
import time
import numpy as np
import shutil

In [2]:
import kagglehub
import shutil
import os

# Download dataset (goes to kagglehub cache)
src_path = kagglehub.dataset_download("gpiosenka/birdies")

# Destination: current directory
dst_path = os.path.join(os.getcwd(), "birdies")

# Copy dataset to current directory
if not os.path.exists(dst_path):
    shutil.copytree(src_path, dst_path)

print("Dataset copied to:", dst_path)

100%|██████████| 736M/736M [00:42<00:00, 18.2MB/s] 


Extracting files...
Dataset copied to: /home/adishesh/Real-Time-Bird-Species-Detection/birdies


In [3]:
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

In [4]:
import random

ROOT = "birdies"
IMAGES_DIR = os.path.join(ROOT, "images")
LABELS_DIR = os.path.join(ROOT, "labels")
TEST_IMAGES_DIR = os.path.join(ROOT, "test images")

# Create split folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(IMAGES_DIR, split), exist_ok=True)
    os.makedirs(os.path.join(LABELS_DIR, split), exist_ok=True)

# Collect all image files (excluding folders)
images = [
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

random.shuffle(images)

n = len(images)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

train_imgs = images[:n_train]
val_imgs   = images[n_train:n_train + n_val]
test_imgs  = images[n_train + n_val:]

def move_pair(img, split):
    img_src = os.path.join(IMAGES_DIR, img)
    lbl_src = os.path.join(LABELS_DIR, img.rsplit(".", 1)[0] + ".txt")

    img_dst = os.path.join(IMAGES_DIR, split, img)
    lbl_dst = os.path.join(LABELS_DIR, split, os.path.basename(lbl_src))

    shutil.move(img_src, img_dst)
    if os.path.exists(lbl_src):
        shutil.move(lbl_src, lbl_dst)

for img in train_imgs:
    move_pair(img, "train")

for img in val_imgs:
    move_pair(img, "val")

for img in test_imgs:
    move_pair(img, "test")

# Handle provided test_images folder (images only)
for img in os.listdir(TEST_IMAGES_DIR):
    if img.lower().endswith((".jpg", ".jpeg", ".png")):
        shutil.move(
            os.path.join(TEST_IMAGES_DIR, img),
            os.path.join(IMAGES_DIR, "test", img)
        )

print("Birdies dataset successfully split!")


Birdies dataset successfully split!


In [ ]:
# -------- SOURCE DATASETS --------
DATASET_1 = "../birdies"
DATASET_2 = "../another-bird-dataset"

# -------- OUTPUT DATASET --------
OUT_ROOT = "combined-birds"

splits_map = {
    "train": "train",
    "val": "valid"
}

In [3]:
for split in ["train", "valid"]:
    os.makedirs(f"{OUT_ROOT}/{split}/images", exist_ok=True)
    os.makedirs(f"{OUT_ROOT}/{split}/labels", exist_ok=True)

In [ ]:
def copy_from_birdies(split, out_split):
    src_images = os.path.join(DATASET_1, "images", split)
    src_labels = os.path.join(DATASET_1, "labels", split)

    dst_images = os.path.join(OUT_ROOT, out_split, "images")
    dst_labels = os.path.join(OUT_ROOT, out_split, "labels")

    for img in os.listdir(src_images):
        if not img.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        label = img.rsplit(".", 1)[0] + ".txt"

        shutil.copy(
            os.path.join(src_images, img),
            os.path.join(dst_images, f"b1_{img}") 
        )

        shutil.copy(
            os.path.join(src_labels, label),
            os.path.join(dst_labels, f"b1_{label}")
        )

In [5]:
def copy_from_another_dataset(split, out_split):
    src_images = os.path.join(DATASET_2, split, "images")
    src_labels = os.path.join(DATASET_2, split, "labels")

    dst_images = os.path.join(OUT_ROOT, out_split, "images")
    dst_labels = os.path.join(OUT_ROOT, out_split, "labels")

    for img in os.listdir(src_images):
        if not img.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        label = img.rsplit(".", 1)[0] + ".txt"

        shutil.copy(
            os.path.join(src_images, img),
            os.path.join(dst_images, f"b2_{img}")
        )

        shutil.copy(
            os.path.join(src_labels, label),
            os.path.join(dst_labels, f"b2_{label}")
        )

In [6]:
copy_from_birdies("train", "train")
copy_from_birdies("val",   "valid")

copy_from_another_dataset("train", "train")
copy_from_another_dataset("valid", "valid")

print("Successfully merged datasets into:", OUT_ROOT)


Successfully merged datasets into: combined-birds


In [ ]:
import yaml
import os

# Get all unique class IDs from labels
class_ids = set()
labels_dir = "../combined-birds/train/labels"

for label_file in os.listdir(labels_dir):
    with open(os.path.join(labels_dir, label_file), 'r') as f:
        for line in f:
            class_id = int(line.split()[0])
            class_ids.add(class_id)

num_classes = max(class_ids) + 1

# Create data.yaml
data_config = {
    'path': os.path.abspath('../combined-birds'),
    'train': 'train/images',
    'val': 'valid/images',
    'nc': num_classes,
    'names': list(range(num_classes))
}

with open('../combined-birds/data.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"data.yaml created with {num_classes} classes")

data.yaml created with 1 classes


In [ ]:
from ultralytics import YOLO
import torch

# Clear GPU cache first
torch.cuda.empty_cache()

# Load pretrained YOLOv8 nano model
model = YOLO("yolov8n.pt")

# Train with metrics
model.train(
    data="combined-birds/data.yaml",
    epochs=20,              
    imgsz=640,               
    batch=16,                
    project="runs",
    name="bird_yolov8n_combined",
    workers=4,      
    cache=True,              # Cache images for faster training
    device=0,       
    amp=True,                # mixed precision training
    patience=20,             # Early stopping
    optimizer='AdamW',       # AdamW optimization
    lr0=0.001,               # Initial learning rate
    lrf=0.01,                # Final learning rate factor
    momentum=0.937,          # Momentum for SGD-like behavior
    weight_decay=0.0005,     # L2 regularization
    warmup_epochs=3,         # Gradual warmup
    warmup_momentum=0.8,     # Starting momentum during warmup
    box=7.5,                 # Box loss gain
    cls=0.5,                 # Class loss gain
    dfl=1.5,                 # Distribution focal loss gain
    hsv_h=0.015,             # HSV-Hue augmentation
    hsv_s=0.7,               # HSV-Saturation augmentation
    hsv_v=0.4,               # HSV-Value augmentation
    degrees=10.0,            # Rotation augmentation
    translate=0.1,           # Translation augmentation
    scale=0.5,               # Scale augmentation
    shear=0.0,               # Shear augmentation
    perspective=0.0,         # Perspective augmentation
    flipud=0.0,              # Vertical flip probability
    fliplr=0.5,              # Horizontal flip probability
    mosaic=1.0,              # Mosaic augmentation probability
    mixup=0.1,               # Mixup augmentation probability
    copy_paste=0    .1,      # Copy-paste augmentation probability
    close_mosaic=10          # Disable mosaic in last N epochs
)

Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=combined-birds/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=bird_yolov8n_combined5, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=20, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x759ea1e88440>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([0.        , 0.001001  , 0.002002  , 0.003003  , 0.004004  ,
       0.00500501, 0.00600601, 0.00700701, 0.00800801, 0.00900901,
       0.01001001, 0.01101101, 0.01201201, 0.01301301, 0.01401401,
       0.01501502, 0.01601602, 0.01701702, 0.01801802, 0.01901902,
       0.02002002, 0.02102102, 0.02202202, 0.02302302, 0.02402402,
       0.02502503, 0.02602603, 0.02702703, 0.02802803, 0.02902903,
       0.03003003, 0.03103103, 0.03203203, 0.03303303, 0.03403403,
       0.03503504, 0.03603604, 0.03703704, 0.03803804, 0.03903904,
       0.04004004, 0.04104104, 0.04204204, 0.04304304, 0.04404404,
       0.04504505, 0.04604605, 0.04704705, 0.04804805, 

In [1]:
from ultralytics import YOLO

model = YOLO("runs/bird_yolov8n_combined5/weights/best.pt")

In [3]:
metrics = model.val(
    data="../combined-birds/data.yaml",
    split="val"
)

precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map5095 = metrics.box.map

print(f"""
YOLOv8 Bird Detection Results
      
===============================
      
Precision     : {precision:.3f}
Recall        : {recall:.3f}
mAP@50        : {map50:.3f}
mAP@50-95     : {map5095:.3f}
""")

Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 298.1±172.4 MB/s, size: 344.6 KB)
val: Scanning /home/adishesh/Real-Time-Bird-Species-Detection/combined-birds/valid/labels.cache... 282 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 282/282 273.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 5.2it/s 3.5s0.1s
                   all        282        425      0.936      0.908      0.934      0.681
Speed: 1.9ms preprocess, 4.2ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /home/adishesh/Real-Time-Bird-Species-Detection/src/runs/detect/val7

YOLOv8 Bird Detection Results


Precision     : 0.936
Recall        : 0.908
mAP@50        : 0.934
mAP@50-95     : 0.681



In [5]:
import numpy as np
import os
import cv2
import time as time
test_dir = "../another-bird-dataset/test/images"
times = []

# Warm-up GPU
for _ in range(10):
    dummy = np.zeros((224, 224, 3), dtype=np.uint8)
    _ = model(dummy, verbose=False)

for img_name in os.listdir(test_dir):
    img = cv2.imread(os.path.join(test_dir, img_name))

    start = time.time()
    _ = model(img, verbose=False)
    times.append(time.time() - start)

avg_time = sum(times) / len(times)
fps = 1 / avg_time

print(f"Avg inference time: {avg_time:.6f}s")
print(f"FPS ceiling: {fps:.2f}")

WARNING ⚠️ 'source' is missing. Using 'source=/home/adishesh/python_env/lib/python3.12/site-packages/ultralytics/assets'.
WARNING ⚠️ 'source' is missing. Using 'source=/home/adishesh/python_env/lib/python3.12/site-packages/ultralytics/assets'.
WARNING ⚠️ 'source' is missing. Using 'source=/home/adishesh/python_env/lib/python3.12/site-packages/ultralytics/assets'.
WARNING ⚠️ 'source' is missing. Using 'source=/home/adishesh/python_env/lib/python3.12/site-packages/ultralytics/assets'.
WARNING ⚠️ 'source' is missing. Using 'source=/home/adishesh/python_env/lib/python3.12/site-packages/ultralytics/assets'.
WARNING ⚠️ 'source' is missing. Using 'source=/home/adishesh/python_env/lib/python3.12/site-packages/ultralytics/assets'.
WARNING ⚠️ 'source' is missing. Using 'source=/home/adishesh/python_env/lib/python3.12/site-packages/ultralytics/assets'.
WARNING ⚠️ 'source' is missing. Using 'source=/home/adishesh/python_env/lib/python3.12/site-packages/ultralytics/assets'.
WARNING ⚠️ 'source' is m

In [6]:
import os

def run_and_save(
    model,
    input_dir,
    output_dir,
    max_images=10,
    conf=0.25,
    iou=0.45,
    max_det=300
):
    os.makedirs(output_dir, exist_ok=True)
    label_dir = os.path.join(output_dir, "labels")
    os.makedirs(label_dir, exist_ok=True)

    images = sorted([
        f for f in os.listdir(input_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    for img_name in images[:max_images]:
        img_path = os.path.join(input_dir, img_name)

        results = model(
            img_path,
            conf=conf,
            iou=iou,
            max_det=max_det,
            verbose=False
        )

        r = results[0]

        # Save image with bounding boxes
        r.save(filename=os.path.join(output_dir, img_name))

        # Save predicted boxes to TXT
        txt_path = os.path.join(
            label_dir, img_name.rsplit(".", 1)[0] + ".txt"
        )

        h, w = r.orig_shape

        with open(txt_path, "w") as f:
            if r.boxes is None:
                continue

            for box in r.boxes:
                cls = int(box.cls.item())
                conf_score = float(box.conf.item())

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                xc = ((x1 + x2) / 2) / w
                yc = ((y1 + y2) / 2) / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h

                f.write(
                    f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f} {conf_score:.4f}\n"
                )

In [7]:
output_dir = "yolo_test_detections"
os.makedirs(output_dir, exist_ok=True)

run_and_save(
    model=model,
    input_dir=test_dir,
    output_dir=output_dir,
    max_images=20,
    conf=0.30,     # detection confidence threshold
    iou=0.6,      # NMS IoU threshold
    max_det=20    # allow many birds per image
)

print("Saved detection images to:", output_dir)

Saved detection images to: yolo_test_detections


In [8]:
video_path = "Video_Generation_Of_Birds_Flying.mp4"

import cv2

cap = cv2.VideoCapture(video_path)

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {width}x{height} @ {fps} FPS, {total_frames} frames")

# Initialize video writer
out = cv2.VideoWriter(
    "output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Run detection
    results = model(frame, verbose=False)

    class_results = classif_model(frame)
    
    # Draw bounding boxes on frame
    bboxed_frame = results[0].plot()

    
    # Write frame to output video
    out.write(bboxed_frame)
    print(class_results[0].item())

    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processed {frame_count}/{total_frames} frames")

# Release resources
cap.release()
out.release()

print(f"Video saved to output.mp4 ({frame_count} frames)")

Video: 0x0 @ 0 FPS, 0 frames
Video saved to output.mp4 (0 frames)


In [11]:
video_path = "sample_inputs/Video_Generation_Of_Birds_Flying.mp4"

import cv2
import time

cap = cv2.VideoCapture(video_path)

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {width}x{height} @ {fps} FPS, {total_frames} frames")
print(f"Real-time requirement: Process each frame in {1/(fps + 1e-6):.4f} seconds")
print("-" * 60)

# Initialize video writer
out = cv2.VideoWriter(
    "output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

frame_count = 0
total_inference_time = 0
inference_times = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Measure inference time
    start_time = time.time()
    results = model(frame, verbose=False)
    bboxed_frame = results[0].plot()
    inference_time = time.time() - start_time
    
    total_inference_time += inference_time
    inference_times.append(inference_time)
    
    # Write frame to output video
    out.write(bboxed_frame)
    
    frame_count += 1
    if frame_count % 30 == 0:
        avg_fps = frame_count / total_inference_time
        current_fps = 1 / inference_time if inference_time > 0 else 0
        real_time_factor = avg_fps / fps
        
        print(f"Frame {frame_count}/{total_frames} | "
              f"Inference: {inference_time*1000:.2f}ms | "
              f"Current FPS: {current_fps:.2f} | "
              f"Avg FPS: {avg_fps:.2f} | "
              f"Real-time: {'✓' if real_time_factor >= 1.0 else '✗'} ({real_time_factor:.2f}x)")

# Release resources
cap.release()
out.release()

# Final statistics
avg_inference_time = total_inference_time / frame_count
avg_fps = frame_count / total_inference_time
real_time_factor = avg_fps / fps

print("\n" + "="*60)
print("PERFORMANCE SUMMARY")
print("="*60)
print(f"Total frames processed: {frame_count}")
print(f"Video FPS: {fps}")
print(f"Average inference time: {avg_inference_time*1000:.2f} ms")
print(f"Average processing FPS: {avg_fps:.2f}")


print(f"\nVideo saved to output.mp4")

Video: 1280x720 @ 24 FPS, 192 frames
Real-time requirement: Process each frame in 0.0417 seconds
------------------------------------------------------------
Frame 30/192 | Inference: 15.77ms | Current FPS: 63.39 | Avg FPS: 52.23 | Real-time: ✓ (2.18x)
Frame 60/192 | Inference: 12.53ms | Current FPS: 79.84 | Avg FPS: 56.53 | Real-time: ✓ (2.36x)
Frame 90/192 | Inference: 18.16ms | Current FPS: 55.07 | Avg FPS: 59.26 | Real-time: ✓ (2.47x)
Frame 120/192 | Inference: 15.89ms | Current FPS: 62.94 | Avg FPS: 60.00 | Real-time: ✓ (2.50x)
Frame 150/192 | Inference: 18.29ms | Current FPS: 54.66 | Avg FPS: 30.04 | Real-time: ✓ (1.25x)
Frame 180/192 | Inference: 15.40ms | Current FPS: 64.95 | Avg FPS: 32.70 | Real-time: ✓ (1.36x)

PERFORMANCE SUMMARY
Total frames processed: 192
Video FPS: 24
Average inference time: 29.78 ms
Average processing FPS: 33.58

Video saved to output.mp4


In [13]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b3

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define number of bird species classes
num_classes = 500

# Load EfficientNet-B3 architecture (no pretrained weights)
species_model = efficientnet_b3(weights=None)

# Modify the classifier head for bird species
in_features = species_model.classifier[1].in_features
species_model.classifier[1] = nn.Linear(in_features, num_classes)

# Load trained weights
species_model.load_state_dict(
    torch.load("models/species_classifier.pth", map_location=device)
)

# Move to device and set to evaluation mode
species_model.to(device)
species_model.eval()

print(f"Species classifier loaded successfully!")
print(f"Model: EfficientNet-B3")
print(f"Classes: {num_classes}")

Using device: cuda
Species classifier loaded successfully!
Model: EfficientNet-B3
Classes: 500


In [14]:
import torch
from torchvision import transforms
from PIL import Image

# Define preprocessing transforms
preprocess = transforms.Compose([
    transforms.Resize((300, 300)),  # EfficientNet-B3 typical input size
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def classify_bird_species(image, model, device):
    """
    Classify a bird crop into species
    
    Args:
        image: PIL Image or numpy array (H, W, C)
        model: EfficientNet species classifier
        device: torch device
    
    Returns:
        class_id: Predicted species class
        confidence: Prediction confidence
    """
    # Convert numpy to PIL if needed
    if isinstance(image, np.ndarray):
        image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    
    # Preprocess
    input_tensor = preprocess(image).unsqueeze(0).to(device)
    
    # Inference
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_class = torch.max(probabilities, 1)
    
    return predicted_class.item(), confidence.item()

# Example usage on a cropped bird image
# bird_crop = frame[y1:y2, x1:x2]  # From YOLO detection
# species_id, conf = classify_bird_species(bird_crop, species_model, device)
# print(f"Species: {species_id}, Confidence: {conf:.2%}")

In [15]:
import cv2
import torch
import time
from torchvision import transforms
from PIL import Image

# Preprocessing for species classifier
preprocess = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

video_path = "sample_inputs/Video_Generation_Of_Birds_Flying.mp4"
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {width}x{height} @ {fps} FPS")

out = cv2.VideoWriter("sample_outputs/output_with_species.mp4", 
                      cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

frame_count = 0
total_time = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    start_time = time.time()
    
    # Step 1: YOLO Detection
    yolo_results = model(frame, verbose=False)
    
    # Step 2: For each detected bird, classify species
    for result in yolo_results:
        boxes = result.boxes
        if boxes is not None:
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                conf = box.conf.item()
                
                # Crop bird region
                bird_crop = frame[y1:y2, x1:x2]
                
                if bird_crop.size > 0:
                    # Classify species
                    bird_pil = Image.fromarray(cv2.cvtColor(bird_crop, cv2.COLOR_BGR2RGB))
                    input_tensor = preprocess(bird_pil).unsqueeze(0).to(device)
                    
                    with torch.no_grad():
                        outputs = species_model(input_tensor)
                        probs = torch.softmax(outputs, dim=1)
                        species_conf, species_id = torch.max(probs, 1)
                    
                    # Draw box and species label
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"Species {species_id.item()}: {species_conf.item():.2%}"
                    cv2.putText(frame, label, (x1, y1-10), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    inference_time = time.time() - start_time
    total_time += inference_time
    
    out.write(frame)
    frame_count += 1
    
    if frame_count % 30 == 0:
        avg_fps = frame_count / total_time
        print(f"Frame {frame_count}/{total_frames} | "
              f"Time: {inference_time*1000:.2f}ms | FPS: {avg_fps:.2f}")

cap.release()
out.release()

print(f"\nVideo with species classification saved!")
print(f"Average FPS: {frame_count/total_time:.2f}")

Video: 1280x720 @ 24 FPS
Frame 30/192 | Time: 126.57ms | FPS: 7.65
Frame 60/192 | Time: 93.72ms | FPS: 8.21
Frame 90/192 | Time: 138.31ms | FPS: 8.56
Frame 120/192 | Time: 110.47ms | FPS: 7.54
Frame 150/192 | Time: 116.34ms | FPS: 7.60
Frame 180/192 | Time: 107.90ms | FPS: 7.74

Video with species classification saved!
Average FPS: 7.83


In [ ]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# Load the image
image_path = "birdies/images/0002.jpg"
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Classify the bird species
species_id, confidence = classify_bird_species(image, species_model, device)

# Display the image with prediction
plt.figure(figsize=(10, 8))
plt.imshow(image_rgb)
plt.axis('off')
plt.title(f"Predicted Species: {species_id} | Confidence: {confidence:.2%}", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Species ID: {species_id}")
print(f"Confidence: {confidence:.4f} ({confidence:.2%})")


In [ ]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# Load the image
image_path = "birdies/images/0002.jpg"
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Classify the bird species
species_id, confidence = classify_bird_species(image, species_model, device)

# Display the image with prediction
plt.figure(figsize=(10, 8))
plt.imshow(image_rgb)
plt.axis('off')
plt.title(f"Predicted Species: {species_id} | Confidence: {confidence:.2%}", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Species ID: {species_id}")
print(f"Confidence: {confidence:.4f} ({confidence:.2%})")
